# Experimento 1: Resampleo de Frecuencias

## ----- FallAllD

### **Problema**

**FallAllD es un problema bloqueante.** Sus 238 Hz no son un múltiplo ni divisor limpio de 100 Hz ni 200 Hz. El resampleo desde 238 Hz a cualquier frecuencia estándar introduce artefactos que se deben caracterizar y reportar.

Para frecuencias estándar, el resampleo es un proceso de división entera. Por ejemplo, pasar de 200 Hz a 100 Hz implica simplemente tomar una de cada dos muestras (y aplicar un filtro paso bajo).

Como 238 Hz no es múltiplo de 100 Hz ni de 200 Hz, el factor de conversión es una fracción (por ejemplo, para llegar a 100 Hz, la relación es $\frac{100}{238} = \frac{50}{119}$). Esto obliga a realizar un **resampleo polifásico**:
1. **Sobremuestrear (Upsampling):** Multiplicar la frecuencia original por 50 (llegando a una frecuencia alta).
2. **Filtrar:** Aplicar un filtro para suavizar los huecos.
3. **Submuestrear (Downsampling):** Dividir esa nueva frecuencia por 119 para llegar finalmente a los 100 Hz.

#### Los artefactos introducidos

El riesgo principal en los datasets de caídas (FallAllD) radica en **los picos de impacto**. Las actividades de la vida diaria (caminar, sentarse) ocurren a bajas frecuencias (generalmente por debajo de $20\text{ Hz}$). Sin embargo, el momento exacto en que una persona golpea el suelo genera un pico de aceleración de alta frecuencia.

Si pasamos de 238 Hz a 100 Hz, la frecuencia de Nyquist (la máxima frecuencia que puedes representar) baja de $119\text{ Hz}$ a $50\text{ Hz}$. El filtro antialiasing necesario para esta conversión podría "suavizar" la amplitud de ese pico de impacto, haciendo que una caída grave se vea como un tropiezo leve en los datos.

**Justificación Analítica**: Asegurar que al transformar la señal, no se elimine información crucial.

### **Hipótesis**

En Python, la biblioteca `SciPy` tiene las herramientas matemáticas exactas para manejar esto de forma óptima sin tener que programar el remuestreo desde cero.

La mejor función para esto es `scipy.signal.resample_poly`, que maneja el proceso de fracciones (up/down) y aplica el filtro FIR antialiasing de manera automática y eficiente.

### **Proceso de Comprobación**

In [ ]:
# Importación de librerías necesarias
%pip install numpy pandas scipy azure-storage-file-datalake

#### 1. Resampleo polifásico
* Importar las librerías
* Realizar el resampleo a la frecuencia: 50 Hz

In [ ]:
import io
import numpy as np
import pandas as pd
from scipy import signal

# Cargar dataset desde Azure
import pandas as pd
from azure.storage.filedatalake import DataLakeServiceClient

# Credenciales
CONNECTION_STRING = "TU_CADENA_DE_CONEXION_AQUI"

# Inicializar el cliente
try:
    service_client = DataLakeServiceClient.from_connection_string(CONNECTION_STRING)
    print("Conexión exitosa con Azure Data Lake.")
except Exception as e:
    print(f"Error al conectar: {e}")

try:
    # Conexión al contenedor y al archivo
    file_system_client_bronce = service_client.get_file_system_client(file_system="bronce")
    directory_client = file_system_client_bronce.get_directory_client("falls")
    file_client_bronce = directory_client.get_file_client("FallAllD-Reduced.csv")

    # Descargar el archivo a la memoria
    download = file_client_bronce.download_file()
    content = download.readall() # en bytes
    
    # Cargar directamente en Pandas usando un buffer de memoria
    df_fallalld = pd.read_csv(io.BytesIO(content))
    print("FallAllD cargado en DataFrame con éxito!!")

except Exception as e:
    print(f"Error en procesamiento de Capa Bronce: {e}")

In [ ]:
# Convertir las señales de aceleración y giroscopio: 238 Hz --> 50 Hz
señal_Ax = df_fallalld['Ax'].values
señal_Ay = df_fallalld['Ay'].values
señal_Az = df_fallalld['Az'].values
señal_Gx = df_fallalld['Gx'].values
señal_Gy = df_fallalld['Gy'].values
señal_Gz = df_fallalld['Gz'].values

# Factor para pasar de 238 Hz a 50 Hz (50/238 simplificado es 25/119)
up = 25
down = 119

# Resampleo polifásico
señal_Ax_resamp = signal.resample_poly(señal_Ax, up, down)
señal_Ay_resamp = signal.resample_poly(señal_Ay, up, down)
señal_Az_resamp = signal.resample_poly(señal_Az, up, down)
señal_Gx_resamp = signal.resample_poly(señal_Gx, up, down)
señal_Gy_resamp = signal.resample_poly(señal_Gy, up, down)
señal_Gz_resamp = signal.resample_poly(señal_Gz, up, down)

#### 2. Superposición visual del impacto
* Aislar una ventana de 2 segundos alrededor de una caída aleatoria en FallAllD.
* Graficar la señal original (238 Hz) y superponer la señal resampleada (100 Hz o 200 Hz).
* Se buscas comprobar visualmente que la amplitud máxima del impacto (el pico) no se haya aplanado significativamente.

#### 3. Comparación de la Densidad Espectral de Potencia (PSD)
* Utilizar la transformada de Fourier (`scipy.signal.welch`) para ver la energía de las señales en el dominio de la frecuencia.
* Esto permitirá demostrar que la información por debajo de $50\text{ Hz}$ se mantiene intacta antes y después de la transformación.

#### Métrica de error máximo
* Calcular la diferencia porcentual entre el valor máximo (pico) de aceleración original y el resampleado.
* Si la diferencia es marginal (ej. $< 5\%$), se justifica cuantitativamente seguir adelante.